In [1]:
import os, re, json, csv, random, numpy as np
import platform, time
import torch
from collections import Counter
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
import subprocess, transformers, bitsandbytes, accelerate
import medmnist
from medmnist import INFO
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, confusion_matrix

from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
login(token=os.getenv("HF_TOKEN"))

MODEL_ID = "Qwen/Qwen3.5-4B"
FLAG = "dermamnist"
RUN = f"{FLAG}_qwen35-4b_4bit"
SIZE = 224
SEED = 42
MAX_NEW_TOKENS = 24   # only valid with thinking disabled
LIMIT = None

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

INFO_D = INFO[FLAG]
CLASS_NAMES = [INFO_D["label"][str(i)] for i in range(len(INFO_D["label"]))]
N_CLASSES = len(CLASS_NAMES)

W0726 23:05:22.951000 22904 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
def compute_metrics(y_true, y_pred, class_names=None, n_classes=None):
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    if n_classes is None:
        n_classes = int(max(y_true.max(), y_pred.max())) + 1
    labels = list(range(n_classes))

    p_mac, r_mac, f_mac, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", labels=labels, zero_division=0)
    p_w, r_w, f_w, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", labels=labels, zero_division=0)
    p_c, r_c, f_c, support = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": p_mac, "recall_macro": r_mac, "f1_macro": f_mac,
        "precision_weighted": p_w, "recall_weighted": r_w, "f1_weighted": f_w,
        "per_class": {"precision": p_c.tolist(), "recall": r_c.tolist(),
                      "f1": f_c.tolist(), "support": support.tolist()},
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
        "class_names": list(class_names) if class_names else None,
    }

def print_metrics(results, title=""):
    if title: print(f"\n--- {title} ---")
    for k in ["accuracy", "balanced_accuracy", "precision_macro",
              "recall_macro", "f1_macro", "f1_weighted"]:
        print(f"{k:<20}: {results[k]:.4f}")
    names, pc = results.get("class_names"), results["per_class"]
    print(f"\n{'class':<45}{'prec':>7}{'rec':>8}{'f1':>8}{'n':>7}")
    for i in range(len(pc["f1"])):
        label = names[i] if names else str(i)
        print(f"{label[:44]:<45}{pc['precision'][i]:>7.3f}"
              f"{pc['recall'][i]:>8.3f}{pc['f1'][i]:>8.3f}{pc['support'][i]:>7}")
    print("\nConfusion matrix (rows = true, cols = pred):")
    for row in results["confusion_matrix"]:
        print("  " + " ".join(f"{v:>5}" for v in row))

def majority_baseline(y_true, class_names=None, n_classes=None):
    y_true = np.asarray(y_true).flatten()
    y_pred = np.full_like(y_true, np.bincount(y_true).argmax())
    return compute_metrics(y_true, y_pred, class_names, n_classes)

In [3]:
DataClass = getattr(medmnist, INFO_D["python_class"])

test_ds = DataClass(split="test", download=True, size=SIZE, as_rgb=True)
train_ds = DataClass(split="train", download=True, size=SIZE, as_rgb=True)

test_labels = np.asarray(test_ds.labels).flatten()
train_labels = np.asarray(train_ds.labels).flatten()

print(f"test: {len(test_ds)}  train: {len(train_ds)}")
print("test class counts:", Counter(test_labels.tolist()))
print_metrics(majority_baseline(test_labels, CLASS_NAMES, N_CLASSES),
              "majority-class baseline (test)")

test: 2005  train: 7007
test class counts: Counter({5: 1341, 4: 223, 2: 220, 1: 103, 0: 66, 6: 29, 3: 23})

--- majority-class baseline (test) ---
accuracy            : 0.6688
balanced_accuracy   : 0.1429
precision_macro     : 0.0955
recall_macro        : 0.1429
f1_macro            : 0.1145
f1_weighted         : 0.5361

class                                           prec     rec      f1      n
actinic keratoses and intraepithelial carcin   0.000   0.000   0.000     66
basal cell carcinoma                           0.000   0.000   0.000    103
benign keratosis-like lesions                  0.000   0.000   0.000    220
dermatofibroma                                 0.000   0.000   0.000     23
melanoma                                       0.000   0.000   0.000    223
melanocytic nevi                               0.669   1.000   0.802   1341
vascular lesions                               0.000   0.000   0.000     29

Confusion matrix (rows = true, cols = pred):
      0     0     0     

In [4]:
"""
used 4-bit NF4 quantization (bitsandbytes) that uses ~3GB VRAM for weights, 
and leaves room for the 256 image tokens + KV cache
"""

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID, 
    quantization_config=bnb, 
    device_map="auto",
    dtype=torch.bfloat16)
model.eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)

print(next(model.parameters()).device, next(model.parameters()).dtype)
print(f"VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

C:\Users\Kawshik\anaconda3\envs\dl_cv\lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


cuda:0 torch.bfloat16
VRAM allocated: 3.30 GB


In [5]:
LETTERS = ["A", "B", "C", "D", "E", "F", "G"]
OPTIONS = "\n".join(f"{L}. {n}" for L, n in zip(LETTERS, CLASS_NAMES))

INSTRUCTION = (
    "You are shown a dermatoscopic image of a skin lesion.\n"
    "Classify it into exactly one of the following categories:\n\n"
    f"{OPTIONS}\n\n"
    "Respond with only the single letter of the correct category (A-G). Do not provide any explanation or extra text."
)

SYNONYMS = {
    0: [r"actinic keratos", r"intraepithelial carcinoma", r"\bakiec\b", r"\bbowen"],
    1: [r"basal cell carcinoma", r"\bbcc\b"],
    2: [r"benign keratosis", r"keratosis-like", r"seborrheic keratosis",
        r"solar lentigo", r"lichen planus-like", r"\bbkl\b"],
    3: [r"dermatofibroma", r"\bdf\b"],
    4: [r"\bmelanomas?\b", r"\bmel\b"],
    5: [r"melanocytic nev", r"\bnev(us|i)\b", r"\bmole\b", r"\bnv\b"],
    6: [r"vascular lesion", r"angioma", r"angiokeratoma", r"pyogenic granuloma", r"\bvasc\b"],
}

def parse_answer(text):
    """Return class index, or -1 if unparseable."""
    t = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    m = re.match(r"^[\s\*\"'`]*\(?([A-G])[\)\.\:,\*]?(\s|$)", t, re.I)
    if m:
        return LETTERS.index(m.group(1).upper())
    m = re.search(r"\*\*([A-G])[\.\)\:]", t)
    if m:
        return LETTERS.index(m.group(1).upper())
    low = t.lower()
    for i, name in enumerate(CLASS_NAMES):
        if name.lower() in low:
            return i
    hits = {i for i, syns in SYNONYMS.items() if any(re.search(s, low) for s in syns)}
    if len(hits) == 1:
        return hits.pop()
    return -1

In [6]:
# for few-shot
rng = np.random.RandomState(SEED)
shot_idx = [int(rng.choice(np.flatnonzero(train_labels == c))) for c in range(N_CLASSES)]
print("exemplar indices:", shot_idx, "labels:", train_labels[shot_idx].tolist())

def build_messages(image, few_shot=False):
    msgs = []
    if few_shot:
        for si in shot_idx:
            img, lab = train_ds[si]
            c = int(np.asarray(lab).flatten()[0])
            msgs.append({"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": INSTRUCTION}]})
            msgs.append({"role": "assistant", "content": [
                {"type": "text", "text": LETTERS[c]}]})
    msgs.append({"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": INSTRUCTION}]})
    return msgs

exemplar indices: [2945, 6797, 2558, 5937, 6261, 4642, 5922] labels: [0, 1, 2, 3, 4, 5, 6]


In [7]:
# inference 
@torch.inference_mode()
def predict_one(image, few_shot=False):
    msgs = build_messages(image, few_shot)
    
    inputs = processor.apply_chat_template(
        msgs, 
        add_generation_prompt=True,
        enable_thinking=False,
        tokenize=True, 
        return_dict=True, 
        return_tensors="pt"
    ).to(model.device)
    
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
    n_in = inputs["input_ids"].shape[-1]
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return processor.decode(out[0][n_in:], skip_special_tokens=True)

In [8]:
# # diagnostic

# img, lab = test_ds[0]
# txt = predict_one(img, few_shot=False)
# print(repr(txt), "->", parse_answer(txt), "| true", int(lab))

# s = processor.apply_chat_template(build_messages(test_ds[0][0]),
#     add_generation_prompt=True, enable_thinking=False, tokenize=False)
# print(repr(s[-120:]))
# print("=========================")

# for c in range(N_CLASSES):
#     i = int(np.flatnonzero(test_labels == c)[0])
#     img, lab = test_ds[i]
#     txt = predict_one(img, few_shot=False)
#     print(f"true={c} raw={txt!r} parsed={parse_answer(txt)}")

In [9]:
def score(preds, y_true, tag):
    preds = np.asarray(preds); y_true = np.asarray(y_true)
    ok = preds != -1
    n_fail = int((~ok).sum())
    print(f"\n===== {tag} =====")
    print(f"parse failures: {n_fail}/{len(preds)} ({100*n_fail/len(preds):.2f}%)")
    res = compute_metrics(y_true[ok], preds[ok], CLASS_NAMES, N_CLASSES)
    print_metrics(res, tag)
    res["n_parse_fail"] = n_fail
    res["n_total"] = int(len(preds))
    print("pred distribution:", Counter(preds[ok].tolist()))
    return res

In [10]:
def run_eval(few_shot=False, limit=None, stop_after_fails=None, dump_csv=None):
    n = len(test_ds) if limit is None else min(limit, len(test_ds))
    raw, preds, lat = [], [], []
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    stopped = False
    for i in range(n):
        img, _ = test_ds[i]
        ts = time.perf_counter()
        txt = predict_one(img, few_shot)
        torch.cuda.synchronize()
        lat.append(time.perf_counter() - ts)
        raw.append(txt)
        preds.append(parse_answer(txt))
        if (i + 1) % 250 == 0:
            print(f"  {i+1}/{n}  fails={preds.count(-1)}  "
                  f"{np.mean(lat[-250:]):.2f} s/img", flush=True)
        if stop_after_fails is not None and preds.count(-1) >= stop_after_fails:
            print(f"stopping at i={i}: {preds.count(-1)} parse failures", flush=True)
            stopped = True
            break

    if dump_csv:
        with open(dump_csv, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["idx", "true", "pred", "parsed_ok", "raw"])
            for j, (t, p) in enumerate(zip(raw, preds)):
                w.writerow([j, int(test_labels[j]), p, int(p != -1), t])
        print(f"wrote {dump_csv} ({len(raw)} rows)", flush=True)

    prof = {
        "wall_time_s": time.perf_counter() - t0,
        "latency_mean_s": float(np.mean(lat)),
        "latency_median_s": float(np.median(lat)),
        "latency_p95_s": float(np.percentile(lat, 95)),
        "peak_vram_gb": torch.cuda.max_memory_allocated() / 1e9,
        "n_images": len(raw),
        "stopped_early": stopped,
    }
    return np.array(preds), raw, test_labels[:len(raw)], prof

In [11]:
results = {}
preds, raw, y, prof = run_eval(few_shot=False, limit=None, dump_csv=f"raw_zero-shot_{RUN}.csv")
print(f"elapsed {prof['wall_time_s']/60:.1f} min | "
      f"{prof['latency_mean_s']:.2f} s/img | peak {prof['peak_vram_gb']:.2f} GB")
r = score(preds, y, "zero-shot")
r["profile"] = prof; r["raw"] = raw; r["preds"] = preds.tolist()
results["zero-shot"] = r

import gc; gc.collect(); torch.cuda.empty_cache()

C:\Users\Kawshik\anaconda3\envs\dl_cv\lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  250/2005  fails=0  1.77 s/img
  500/2005  fails=0  1.79 s/img
  750/2005  fails=0  1.82 s/img
  1000/2005  fails=0  1.80 s/img
  1250/2005  fails=0  1.82 s/img
  1500/2005  fails=0  1.83 s/img
  1750/2005  fails=0  1.87 s/img
  2000/2005  fails=0  1.88 s/img
wrote raw_zero-shot_dermamnist_qwen35-4b_4bit.csv (2005 rows)
elapsed 60.9 min | 1.82 s/img | peak 3.44 GB

===== zero-shot =====
parse failures: 0/2005 (0.00%)

--- zero-shot ---
accuracy            : 0.6399
balanced_accuracy   : 0.4212
precision_macro     : 0.4046
recall_macro        : 0.4212
f1_macro            : 0.3146
f1_weighted         : 0.6317

class                                           prec     rec      f1      n
actinic keratoses and intraepithelial carcin   0.257   0.439   0.324     66
basal cell carcinoma                           0.409   0.087   0.144    103
benign keratosis-like lesions                  0.357   0.559   0.435    220
dermatofibroma                                 0.133   0.087   0.105     23
mela

In [12]:
preds, raw, y, prof = run_eval(few_shot=True, limit=None, dump_csv=f"raw_7-shot_{RUN}.csv")
print(f"elapsed {prof['wall_time_s']/60:.1f} min | "
      f"{prof['latency_mean_s']:.2f} s/img | peak {prof['peak_vram_gb']:.2f} GB")
r = score(preds, y, "few-shot")
r["profile"] = prof; r["raw"] = raw; r["preds"] = preds.tolist()
results["few-shot"] = r

import gc; gc.collect(); torch.cuda.empty_cache()

C:\Users\Kawshik\anaconda3\envs\dl_cv\lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  250/2005  fails=0  2.48 s/img
  500/2005  fails=0  2.43 s/img
  750/2005  fails=0  2.45 s/img
  1000/2005  fails=0  2.43 s/img
  1250/2005  fails=0  2.40 s/img
  1500/2005  fails=0  2.42 s/img
  1750/2005  fails=0  2.46 s/img
  2000/2005  fails=0  2.45 s/img
wrote raw_7-shot_dermamnist_qwen35-4b_4bit.csv (2005 rows)
elapsed 81.5 min | 2.44 s/img | peak 3.89 GB

===== few-shot =====
parse failures: 0/2005 (0.00%)

--- few-shot ---
accuracy            : 0.6195
balanced_accuracy   : 0.4499
precision_macro     : 0.3521
recall_macro        : 0.4499
f1_macro            : 0.3175
f1_weighted         : 0.6319

class                                           prec     rec      f1      n
actinic keratoses and intraepithelial carcin   0.169   0.833   0.281     66
basal cell carcinoma                           0.250   0.068   0.107    103
benign keratosis-like lesions                  0.393   0.218   0.281    220
dermatofibroma                                 0.200   0.043   0.071     23
melanoma 

In [13]:
try:
    gpu = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"], text=True).strip()
except Exception:
    gpu = torch.cuda.get_device_name(0)

env = {
    "gpu": gpu,
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "bitsandbytes": bitsandbytes.__version__,
    "accelerate": accelerate.__version__,
    "python": platform.python_version(),
    "platform": platform.platform(),
}

out = f"results_{RUN}.json"
payload = {
    "model": MODEL_ID,
    "dataset": FLAG,
    "quantization": "nf4-4bit-double",
    "compute_dtype": "bfloat16",
    "decoding": "greedy",
    "max_new_tokens": MAX_NEW_TOKENS,
    "thinking": False,
    "batch_size": 1,
    "seed": SEED,
    "image_size_in": SIZE,
    "n_test": int(len(test_ds)),
    "limit": LIMIT,
    "shot_indices": shot_idx,
    "prompt": INSTRUCTION,
    "env": env,
    "baseline": {k: v for k, v in
                 majority_baseline(test_labels, CLASS_NAMES, N_CLASSES).items()
                 if k != "class_names"},
    "conditions": {k: {kk: vv for kk, vv in v.items() if kk != "raw"}
                   for k, v in results.items()},
    "raw_outputs": {k: v["raw"] for k, v in results.items()},
}
with open(out, "w") as f:
    json.dump(payload, f, indent=2)
print("saved", out)

saved results_dermamnist_qwen35-4b_4bit.json


In [14]:
print(f"{'condition':<12}{'f1_macro':>10}{'bal_acc':>10}{'acc':>8}"
      f"{'s/img':>8}{'VRAM':>8}{'fail%':>8}")
b = majority_baseline(test_labels, CLASS_NAMES, N_CLASSES)
print(f"{'majority':<12}{b['f1_macro']:>10.4f}{b['balanced_accuracy']:>10.4f}"
      f"{b['accuracy']:>8.4f}{'-':>8}{'-':>8}{'-':>8}")
for tag, r in results.items():
    p = r["profile"]
    print(f"{tag:<12}{r['f1_macro']:>10.4f}{r['balanced_accuracy']:>10.4f}"
          f"{r['accuracy']:>8.4f}{p['latency_mean_s']:>8.2f}"
          f"{p['peak_vram_gb']:>8.2f}"
          f"{100*r['n_parse_fail']/r['n_total']:>8.2f}")

condition     f1_macro   bal_acc     acc   s/img    VRAM   fail%
majority        0.1145    0.1429  0.6688       -       -       -
zero-shot       0.3146    0.4212  0.6399    1.82    3.44    0.00
few-shot        0.3175    0.4499  0.6195    2.44    3.89    0.00
